Ethan Ho  
CMPE 256  
3/30/26  
Song Recommender System

In [3]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import keras
from keras import layers
import tensorflow as tf
from keras.models import Model
from keras.optimizers import Adam
from keras.layers import Add, Activation, Lambda, BatchNormalization, Concatenate, Dropout, Input, Embedding, Dot, Reshape, Dense, Flatten

In [4]:
df = pd.read_csv("train.csv")
df.tail()

,user_id,song_id,rating
195348,1074598,1007271.0,2.75
195349,1449917,1005001.0,9.50
195350,1032535,1006807.0,4.50
195351,1181294,1132765.0,5.00
195352,1514142,NaN,NaN


Because only the last row has NaNs I remove them with iloc

In [8]:
df = df.iloc[:-1]

In [9]:
x, y = df[["user_id", "song_id"]].values, df["rating"]

In [10]:
# Checking for null values
print(y.isnull().sum())

0


In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.2)
print('Train set ratings: {}'.format(len(y_train)))
print('Test set ratings: {}'.format(len(y_test)))

Train set ratings: 156281
Test set ratings: 39071


In [12]:
# splitting user, item columns for embedding
x_train_arr = [x_train[:, 0], x_train[:, 1]]
x_test_arr = [x_test[:, 0], x_test[:, 1]]

In [46]:
# dimensons of the input arr
user_dim = df["user_id"].max()+1
item_dim = df["song_id"].max()+1
out_dim = 32

In [47]:
# Model
def GMFact(user_dim, item_dim, out_dim, activation):
    # Input Layer
    user = Input(name = 'u_in', shape = [1])
    song = Input(name = 's_in', shape = [1])

    # Embeddings
    song_embedding = Embedding(name = 's_emb',
                       input_dim = item_dim,
                       output_dim = out_dim)(song)

    user_embedding = Embedding(name = 'u_emb',
                       input_dim = user_dim,
                       output_dim = out_dim)(user)

    # Element-wise Multiply Layer
    x = tf.keras.layers.Multiply()([user_embedding, song_embedding])

    x = Flatten()(x)

    # Single Neuron
    x = Dense(1, kernel_initializer='lecun_uniform')(x)
    x = Activation(activation)(x)

    model = Model(inputs=[user, song], outputs=x)

    model.compile(
      optimizer='sgd',
      loss='mse',
      metrics=[tf.keras.metrics.RootMeanSquaredError()])

    return model

Now trying out different activation functions

In [48]:
sigmoid = GMFact(int(user_dim), int(item_dim), out_dim, "sigmoid")
sigmoid = sigmoid.fit(
    x=x_train_arr,
    y=y_train,
    batch_size=50,
    epochs=5,
    verbose=1,
    validation_data=(x_test_arr, y_test)
)

Epoch 1/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 22.2520 - root_mean_squared_error: 4.7172 - val_loss: 22.0779 - val_root_mean_squared_error: 4.6987
Epoch 2/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 22.0677 - root_mean_squared_error: 4.6976 - val_loss: 22.0612 - val_root_mean_squared_error: 4.6969
Epoch 3/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 22.0582 - root_mean_squared_error: 4.6966 - val_loss: 22.0557 - val_root_mean_squared_error: 4.6963
Epoch 4/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 22.0543 - root_mean_squared_error: 4.6962 - val_loss: 22.0530 - val_root_mean_squared_error: 4.6961
Epoch 5/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 22.0522 - root_mean_squared_error: 4.6960 - val_loss: 22.0513 - val_root_mean_squared_error: 4.6959


I randomly chose gelu cause its the most optimal for llms

In [49]:
gelu = GMFact(int(user_dim), int(item_dim), out_dim, "gelu")
gelu = gelu.fit(
    x=x_train_arr,
    y=y_train,
    batch_size=50,
    epochs=5,
    verbose=1,
    validation_data=(x_test_arr, y_test)
)

Epoch 1/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 3.2574 - root_mean_squared_error: 1.8048 - val_loss: 3.0022 - val_root_mean_squared_error: 1.7327
Epoch 2/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0026 - root_mean_squared_error: 1.7328 - val_loss: 3.0022 - val_root_mean_squared_error: 1.7327
Epoch 3/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0024 - root_mean_squared_error: 1.7327 - val_loss: 3.0039 - val_root_mean_squared_error: 1.7332
Epoch 4/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 3.0025 - root_mean_squared_error: 1.7328 - val_loss: 3.0024 - val_root_mean_squared_error: 1.7327
Epoch 5/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0026 - root_mean_squared_error: 1.7328 - val_loss: 3.0024 - val_root_mean_squared_error: 1.7327


In [74]:
relu = GMFact(int(user_dim), int(item_dim), out_dim, "relu")
relu_history = relu.fit(
    x=x_train_arr,
    y=y_train,
    batch_size=50,
    epochs=5,
    verbose=1,
    validation_data=(x_test_arr, y_test)
)

Epoch 1/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 3.2435 - root_mean_squared_error: 1.8010 - val_loss: 3.0029 - val_root_mean_squared_error: 1.7329
Epoch 2/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 3.0022 - root_mean_squared_error: 1.7327 - val_loss: 3.0021 - val_root_mean_squared_error: 1.7327
Epoch 3/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0024 - root_mean_squared_error: 1.7328 - val_loss: 3.0023 - val_root_mean_squared_error: 1.7327
Epoch 4/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0025 - root_mean_squared_error: 1.7328 - val_loss: 3.0023 - val_root_mean_squared_error: 1.7327
Epoch 5/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 3.0024 - root_mean_squared_error: 1.7328 - val_loss: 3.0021 - val_root_mean_squared_error: 1.7327


I also changed the dimensions of the embedding from 8 to 32 but the results remained mostly the same. Now we will try NeuMF based off the provided paper Neural Collaborative Filtering.

In [51]:
def NeuMF(user_dim, item_dim, gmf_dim, mlp_dim):
    user = Input(name='u_in', shape=[1])
    song = Input(name='s_in', shape=[1])

    # GMF path
    u_gmf = Flatten()(Embedding(user_dim, gmf_dim)(user))
    s_gmf = Flatten()(Embedding(item_dim, gmf_dim)(song))
    gmf_out = tf.keras.layers.Multiply()([u_gmf, s_gmf])

    # MLP path
    u_mlp = Flatten()(Embedding(user_dim, mlp_dim)(user))
    s_mlp = Flatten()(Embedding(item_dim, mlp_dim)(song))
    mlp_out = Concatenate()([u_mlp, s_mlp])
    mlp_out = Dense(64, activation='relu')(mlp_out)
    mlp_out = Dropout(0.2)(mlp_out)
    mlp_out = Dense(32, activation='relu')(mlp_out)

    # Combine
    x = Concatenate()([gmf_out, mlp_out])
    x = Dense(1, activation='relu')(x)

    model = Model(inputs=[user, song], outputs=x)
    model.compile(optimizer=Adam(0.001), loss='mse',
                  metrics=[tf.keras.metrics.RootMeanSquaredError()])
    return model

In [53]:
neuRELU = NeuMF(int(user_dim), int(item_dim), gmf_dim=32, mlp_dim=32)
neuRELU = neuRELU.fit(
    x=x_train_arr,
    y=y_train,
    batch_size=50,
    epochs=5,
    verbose=1,
    validation_data=(x_test_arr, y_test)
)

Epoch 1/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 123s 38ms/step - loss: 3.6044 - root_mean_squared_error: 1.8985 - val_loss: 2.9865 - val_root_mean_squared_error: 1.7281
Epoch 2/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 116s 37ms/step - loss: 2.2974 - root_mean_squared_error: 1.5157 - val_loss: 3.0904 - val_root_mean_squared_error: 1.7579
Epoch 3/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 117s 37ms/step - loss: 0.8080 - root_mean_squared_error: 0.8989 - val_loss: 3.1742 - val_root_mean_squared_error: 1.7816
Epoch 4/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 116s 37ms/step - loss: 0.3106 - root_mean_squared_error: 0.5573 - val_loss: 3.1911 - val_root_mean_squared_error: 1.7864
Epoch 5/5
3126/3126 ━━━━━━━━━━━━━━━━━━━━ 119s 38ms/step - loss: 0.2293 - root_mean_squared_error: 0.4789 - val_loss: 3.1947 - val_root_mean_squared_error: 1.7874


I started with matrix factorization because I believed in back propagation and the latent factors more than ranking based off similar users or items. However, I was atleast going to try both methods until I see that neuRELU has such a low RMSE

In [55]:
print(neuRELU.history['val_root_mean_squared_error'][-1])

1.7873713970184326


Nevermind, it seems to be super overfitted and performs a little worse than the basic matrix factorization. Now trying basic collaborative filtering based off assignment 3

In [60]:
import math
from collections import defaultdict

In [61]:
dataset = df.to_dict('records')

In [63]:
# Utilities data strutures
usersPerItem = defaultdict(set)
itemsPerUser = defaultdict(set)
ratingDict = {}

for d in dataset:
    user, item = d['user_id'], d['song_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    ratingDict[(user, item)] = d['rating']

reviewsPerUser = defaultdict(list)
reviewsPerItem = defaultdict(list)
for d in dataset:
    user, item = d['user_id'], d['song_id']
    reviewsPerUser[user].append(d)
    reviewsPerItem[item].append(d)

In [64]:
# Averages
userAverages = {}
itemAverages = {}

for u in itemsPerUser:
    rs = [ratingDict[(u, i)] for i in itemsPerUser[u]]
    userAverages[u] = sum(rs) / len(rs)

for i in usersPerItem:
    rs = [ratingDict[(u, i)] for u in usersPerItem[i]]
    itemAverages[i] = sum(rs) / len(rs)

ratingMean = sum(d['rating'] for d in dataset) / len(dataset)

In [65]:
# Similarity Metric
def PearsonUser(u1, u2):
    inter = itemsPerUser[u1].intersection(itemsPerUser[u2])
    if not inter:
        return 0

    u1Bar = userAverages[u1]
    u2Bar = userAverages[u2]

    numer, denom1, denom2 = 0, 0, 0
    for i in inter:
        r1 = ratingDict[(u1, i)] - u1Bar
        r2 = ratingDict[(u2, i)] - u2Bar
        numer += r1 * r2
        denom1 += r1 ** 2
        denom2 += r2 ** 2

    denom = math.sqrt(denom1) * math.sqrt(denom2)
    return numer / denom if denom != 0 else 0

In [66]:
# Prediction
def predictRating(user, item):
    ratings = []
    similarities = []
    for d in reviewsPerItem[item]:
        u2 = d['user_id']
        if u2 == user:
            continue
        sim = PearsonUser(user, u2)
        if sim <= 0:
            continue
        ratings.append(d['rating'] - userAverages[u2])
        similarities.append(sim)
    if sum(similarities) > 0:
        weightedRatings = [x * y for x, y in zip(ratings, similarities)]
        return userAverages[user] + (sum(weightedRatings) / sum(similarities))
    else:
        return ratingMean

In [68]:
def MSE(predictions, labels):
    differences = [(x - y) ** 2 for x, y in zip(predictions, labels)]
    return sum(differences) / len(differences)

def RMSE(predictions, labels):
    return math.sqrt(MSE(predictions, labels))

labels = [d['rating'] for d in dataset]
alwaysPredictMean = [ratingMean for d in dataset]

simPredictions = [predictRating(d['user_id'], d['song_id']) for d in dataset]

print(f"Mean baseline MSE:      {RMSE(alwaysPredictMean, labels):.4f}")
print(f"Pearson user-based RMSE:{RMSE(simPredictions, labels):.4f}")

Mean baseline MSE:      1.7326
Pearson user-based RMSE:1.6017


Dispute my confidence in matrix factorization, the simple user based collaborative filtering performed slightly better. I will likely try submitting both the user-based CF and reluMF to see which would give the lower RMSE.

In [77]:
submission_df = pd.read_csv("test.csv")

predictions = []
for _, row in submission_df.iterrows():
    user, item = row['user_id'], row['song_id']
    pred = predictRating(user, item)
    predictions.append({
        'user_id-song_id': f"{user}-{item}",
        'rating': round(pred, 2)
    })

output_df = pd.DataFrame(predictions)
output_df.to_csv("cf_predictions.csv", index=False)
print(output_df.head())

   user_id-song_id  rating
0  1717534-1005189    5.36
1  1302257-1042789    5.36
2  1700269-1042495    5.36
3  1265736-1040200    5.36
4  1060963-1008334    5.36


In [78]:
test_df = pd.read_csv("test.csv")

x_test_final = [test_df['user_id'].values, test_df['song_id'].values]
preds = relu.predict(x_test_final).flatten()

output_df = pd.DataFrame({
    'user_id-song_id': test_df['user_id'].astype(str) + '-' + test_df['song_id'].astype(str),
    'rating': preds.round(2)
})

output_df.to_csv("mf_predictions.csv", index=False)
print(output_df.head())

33239/33239 ━━━━━━━━━━━━━━━━━━━━ 47s 1ms/step
   user_id-song_id  rating
0  1717534-1005189    5.36
1  1302257-1042789    5.36
2  1700269-1042495    5.36
3  1265736-1040200    5.37
4  1060963-1008334    5.36
